In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Check the SageMaker instance and kernel.")

DEVICE = "cuda"
print(torch.cuda.get_device_name(0))

# RQ1: batch-32 prior-mixing curve, 3 seeds

## 1. Environment and setup

This section locates the project and records the Python, PyTorch, and compute device used for the experiment.

It does not install packages, download data, or start training. If an import fails, fix the notebook's Python environment before continuing.


In [ ]:
from pathlib import Path
import sys
import torch

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").is_file() and (path / "src" / "tfmprior").is_dir():
            return path
    raise FileNotFoundError("Open this notebook from inside the PriorResearch repository.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")


## 2. Experiment settings

**Research question:** How does the finance-prior share affect performance on finance and general tabular benchmarks?

We vary the finance share `r` and training seed. Model architecture and training settings stay fixed across runs, giving 5 shares × 3 seeds = 15 runs.

In [ ]:
FINANCE_SHARES = [0.0, 0.3, 0.5, 0.7, 1.0]
SEEDS = [0, 1, 2]

TRAINING = {
    "steps": 3000,
    "batch_size": 32,
    "lr": 0.004,
}

MODEL = {
    "embedding_size": 96,
    "num_attention_heads": 4,
    "mlp_hidden_size": 192,
    "num_layers": 3,
    "num_outputs": 2,
}

SPLIT_SIZES = list(range(80, 225, 16))

EXPERIMENT_NAME = "rq1_bs32_curve_v1"

N_RUNS = len(FINANCE_SHARES) * len(SEEDS)

EVAL_SUITES = ["finance", "general"]
N_FOLDS = 5

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Finance shares: {FINANCE_SHARES}")
print(f"Seeds: {SEEDS}")
print(f"Planned training runs: {N_RUNS}")
print(f"Training settings: {TRAINING}")

## 3. Check the input data

The training pools contain synthetic pretraining tables. Separate 1k-table pools are reserved for synthetic validation.

The benchmark cache contains the fixed real-table samples used for evaluation. This cell only checks that the required files exist; it does not download data or train a model.

In [ ]:
from tfmprior.eval.benchmarks import SUITES as SUITE_TASKS

POOLS_DIR = PROJECT_ROOT / "artifacts" / "dumps"
BENCHMARK_CACHE_DIR = PROJECT_ROOT / "artifacts" / "benchmark_cache"

TRAIN_POOLS = {
    "graph_scm": "generic_graph_scm_100k_256x24_seed0.h5",
    "fintfm": "finance_100k_256x24_seed0.h5",
}

VALIDATION_POOLS = {
    "graph_scm": "generic_graph_scm_1k_256x24_seed1.h5",
    "fintfm": "finance_1k_256x24_seed1.h5",
}

required_pool_paths = []

for filename in TRAIN_POOLS.values():
    required_pool_paths.append(POOLS_DIR / filename)

for filename in VALIDATION_POOLS.values():
    required_pool_paths.append(POOLS_DIR / filename)


required_cache_paths = []

for suite_tasks in SUITE_TASKS.values():
    for task_id, dataset_name in suite_tasks.items():
        filename = f"{task_id}_{dataset_name}.npz"
        required_cache_paths.append(BENCHMARK_CACHE_DIR / filename)

missing_pools = [path for path in required_pool_paths if not path.is_file()]
missing_cache = [path for path in required_cache_paths if not path.is_file()]

print(f"Prior pools: {len(required_pool_paths) - len(missing_pools)} / {len(required_pool_paths)} found")
print(f"Benchmark cache files: {len(required_cache_paths) - len(missing_cache)} / {len(required_cache_paths)} found")

missing_files = missing_pools + missing_cache
if missing_files:
    missing_list = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(f"Required experiment inputs are missing:\n{missing_list}")

print("All required input files are present.")

## 4. Baseline models

We evaluate Random Forest and Logistic Regression on the finance and general benchmarks using stratified five-fold cross-validation.

These models provide reference scores for the prior-mixing experiments.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

baseline_models = {
    "random_forest": make_pipeline(
        SimpleImputer(strategy="mean", keep_empty_features=True),
        RandomForestClassifier(
            n_estimators=200,
            random_state=0,
            n_jobs=1,
        ),
    ),
    "logistic_regression": make_pipeline(
        SimpleImputer(strategy="mean", keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    ),
}

In [ ]:
import numpy as np

finance_task_ids = {
    363694, 363706, 363626, 363627,
    363673, 363676, 363682,
}

benchmark_suites = {
    "finance": {},
    "general": {},
}

for path in sorted(BENCHMARK_CACHE_DIR.glob("*.npz")):
    task_id, dataset_name = path.stem.split("_", maxsplit=1)

    with np.load(path, allow_pickle=False) as data:
        if "skipped" in data:
            continue

        samples = [
            (data[f"X{i}"].copy(), data[f"y{i}"].copy())
            for i in range(3)
        ]

    suite_name = (
        "finance"
        if int(task_id) in finance_task_ids
        else "general"
    )
    benchmark_suites[suite_name][dataset_name] = samples

for suite_name, datasets in benchmark_suites.items():
    print(f"{suite_name}: {len(datasets)} datasets")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold


def predict_out_of_fold(model, X, y, n_folds=5):
    probabilities = np.zeros(len(y))

    folds = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=0,
    )

    for train_indices, test_indices in folds.split(X, y):
        fold_model = clone(model)

        fold_model.fit(X[train_indices], y[train_indices])

        probabilities[test_indices] = fold_model.predict_proba(
            X[test_indices]
        )[:, 1]

    return probabilities

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss


def expected_calibration_error(y, p, n_bins=10):
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    ece = 0.0

    for b in range(n_bins):
        in_bin = bins == b

        if in_bin.any():
            predicted_rate = p[in_bin].mean()
            observed_rate = y[in_bin].mean()
            weight = in_bin.mean()

            ece += weight * abs(predicted_rate - observed_rate)

    return float(ece)


def binary_metrics(y, p):
    return {
        "auc": float(roc_auc_score(y, p)),
        "ece": expected_calibration_error(y, p),
        "brier": float(brier_score_loss(y, p)),
    }

In [ ]:
import pandas as pd

baseline_rows = []

for model_name, model in baseline_models.items():
    for suite_name, datasets in benchmark_suites.items():
        for dataset_name, samples in datasets.items():
            for repeat, (X, y) in enumerate(samples):
                p = predict_out_of_fold(
                    model, X, y, n_folds=N_FOLDS
                )
                scores = binary_metrics(y, p)

                baseline_rows.append({
                    "model": model_name,
                    "suite": suite_name,
                    "dataset": dataset_name,
                    "repeat": repeat,
                    **scores,
                })

        print(f"Finished: {model_name} / {suite_name}", flush=True)

baseline_results = pd.DataFrame(baseline_rows)
baseline_results.head()

In [ ]:
metrics = ["auc", "ece", "brier"]

# Average the three repeats within each dataset.
baseline_per_dataset = (
    baseline_results
    .groupby(["model", "suite", "dataset"], as_index=False)[metrics]
    .mean()
)

# Give each dataset equal weight within its suite.
baseline_summary = (
    baseline_per_dataset
    .groupby(["model", "suite"], as_index=False)[metrics]
    .mean()
)

baseline_summary.round(4)

## 5. Training plan

Train one nanoTabPFN model for each finance-prior share and seed. Model architecture, training budget, and evaluation protocol are fixed across runs.

In [ ]:
run_plan = []

for r in FINANCE_SHARES:
    for seed in SEEDS:
        run_plan.append({
            "finance_share": r,
            "generic_share": 1.0 - r,
            "seed": seed,
        })

training_plan = pd.DataFrame(run_plan)
tables_per_run = TRAINING["steps"] * TRAINING["batch_size"]

print(f"Training runs: {len(training_plan)}")
print(f"Table draws per run: {tables_per_run:,}")

training_plan

In [ ]:
import random
from tfmprior.model.nanotabpfn import NanoTabPFNModel


def create_model(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = NanoTabPFNModel(**MODEL)
    return model.to(DEVICE)

In [ ]:
from tfmprior.data.loader import PoolMixtureLoader


def create_loader(r, seed):
    mixture = {
        "graph_scm": 1.0 - r,
        "fintfm": r,
    }

    pool_paths = {
        name: str(POOLS_DIR / filename)
        for name, filename in TRAIN_POOLS.items()
    }

    return PoolMixtureLoader(
        mixture=mixture,
        pools=pool_paths,
        num_steps=TRAINING["steps"],
        batch_size=TRAINING["batch_size"],
        split_sizes=SPLIT_SIZES,
        seed=seed,
    )

## 6. Train and evaluate

Each run starts from a freshly initialized model and trains on its specified prior mixture. Record the training loss and evaluate the final model on the benchmark suites.

In [ ]:
from tfmprior.train.loop import train


def train_one_run(r, seed):
    model = create_model(seed)
    loader = create_loader(r, seed)
    training_rows = []

    def record_step(step, loss, batch, seconds):
        training_rows.append({
            "step": step,
            "loss": loss,
            "seconds": seconds,
        })

        if step % 100 == 0 or step == TRAINING["steps"]:
            print(
                f"r={r:g}, seed={seed}, "
                f"step={step}, loss={loss:.4f}",
                flush=True,
            )

    model = train(
        model=model,
        loader=loader,
        lr=TRAINING["lr"],
        device=DEVICE,
        on_step=record_step,
    )

    return model, pd.DataFrame(training_rows)

In [ ]:
from tfmprior.model.nanotabpfn import NanoTabPFNClassifier


def predict_pretrained_out_of_fold(model, X, y, n_folds=5):
    model.eval()
    probabilities = np.zeros(len(y))

    folds = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=0,
    )

    for train_indices, test_indices in folds.split(X, y):
        imputer = SimpleImputer(
            strategy="mean",
            keep_empty_features=True,
        )

        X_train = imputer.fit_transform(X[train_indices])
        X_test = imputer.transform(X[test_indices])

        classifier = NanoTabPFNClassifier(model, DEVICE)
        classifier.fit(X_train, y[train_indices])

        probabilities[test_indices] = classifier.predict_proba(
            X_test
        )[:, 1]

    return probabilities

In [ ]:
def evaluate_model(model, r, seed):
    result_rows = []

    for suite_name, datasets in benchmark_suites.items():
        for dataset_name, samples in datasets.items():
            for repeat, (X, y) in enumerate(samples):
                p = predict_pretrained_out_of_fold(
                    model, X, y, n_folds=N_FOLDS
                )
                scores = binary_metrics(y, p)

                result_rows.append({
                    "finance_share": r,
                    "seed": seed,
                    "suite": suite_name,
                    "dataset": dataset_name,
                    "repeat": repeat,
                    **scores,
                })

        print(
            f"Evaluated: r={r:g}, seed={seed}, suite={suite_name}",
            flush=True,
        )

    return pd.DataFrame(result_rows)

In [ ]:
import gc

all_run_results = []

for run in run_plan:
    r = run["finance_share"]
    seed = run["seed"]

    run_dir = (
        PROJECT_ROOT / "runs" / EXPERIMENT_NAME
        / f"r{r:g}_seed{seed}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = run_dir / "checkpoint.pt"
    results_path = run_dir / "results.csv"

    run_config = {
        "model": MODEL,
        "training": TRAINING,
        "split_sizes": SPLIT_SIZES,
        "finance_share": r,
        "seed": seed,
        "n_folds": N_FOLDS,
    }

    print(f"\nStarting: r={r:g}, seed={seed}", flush=True)

    if checkpoint_path.is_file():
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True,
        )

        if checkpoint["config"] != run_config:
            raise ValueError(
                f"Settings changed for {run_dir}. Use a new EXPERIMENT_NAME."
            )

        if results_path.is_file():
            all_run_results.append(pd.read_csv(results_path))
            print("Already completed; skipping.", flush=True)
            continue

        model = create_model(seed)
        model.load_state_dict(checkpoint["model"])
        del checkpoint

    else:
        model, training_log = train_one_run(r, seed)

        temporary_checkpoint = run_dir / "checkpoint.tmp"
        torch.save(
            {
                "model": model.state_dict(),
                "config": run_config,
            },
            temporary_checkpoint,
        )
        temporary_checkpoint.replace(checkpoint_path)

        training_log.to_csv(
            run_dir / "training_log.csv",
            index=False,
        )

    results = evaluate_model(model, r, seed)

    temporary_results = run_dir / "results.tmp.csv"
    results.to_csv(temporary_results, index=False)
    temporary_results.replace(results_path)

    all_run_results.append(results)

    del model
    gc.collect()
    torch.cuda.empty_cache()

experiment_results = pd.concat(all_run_results, ignore_index=True)

## 7. Results and comparison

Average the three sampling repeats within each dataset, then average datasets within each suite. Keep training seeds separate before summarizing variation across seeds.

In [ ]:
metrics = ["auc", "ece", "brier"]

model_per_dataset = (
    experiment_results
    .groupby(
        ["finance_share", "seed", "suite", "dataset"],
        as_index=False,
    )[metrics]
    .mean()
)

model_per_suite = (
    model_per_dataset
    .groupby(
        ["finance_share", "seed", "suite"],
        as_index=False,
    )[metrics]
    .mean()
)

model_per_suite.round(4)

In [ ]:
curve_summary = (
    model_per_suite
    .groupby(["finance_share", "suite"])[metrics]
    .agg(["mean", "std"])
)

curve_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in curve_summary.columns
]

curve_summary = curve_summary.reset_index()

curve_summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, suite in enumerate(EVAL_SUITES):
    curve = (
        curve_summary[curve_summary["suite"] == suite]
        .sort_values("finance_share")
    )
    baselines = baseline_summary[
        baseline_summary["suite"] == suite
    ]

    for col, metric in enumerate(metrics):
        ax = axes[row, col]

        ax.errorbar(
            curve["finance_share"],
            curve[f"{metric}_mean"],
            yerr=curve[f"{metric}_std"],
            marker="o",
            capsize=4,
            label="nanoTabPFN",
        )

        for _, baseline in baselines.iterrows():
            ax.axhline(
                baseline[metric],
                linestyle="--",
                label=baseline["model"],
            )

        direction = "higher is better" if metric == "auc" else "lower is better"
        ax.set_title(f"{suite.capitalize()} — {metric.upper()}")
        ax.set_xlabel("Finance prior share r")
        ax.set_ylabel(f"{metric.upper()} ({direction})")
        ax.set_xticks(FINANCE_SHARES)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

fig.tight_layout()
plt.show()